 Wasn't sure what I was doing, was just trying to visualize correlation between leasing value and other fields and remove NaN values

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

leaseData = pd.read_csv("Leases.csv")
occupancyData = pd.read_csv("Major Market Occupancy Data-revised-20250412-051645.csv")

In [17]:
#what data are we working with
lDc = leaseData.columns
#quantify NaN values
nan_counts = leaseData.isna().sum()
lDc, leaseData.shape

(Index(['year', 'quarter', 'monthsigned', 'market', 'building_name',
        'building_id', 'address', 'region', 'city', 'state', 'zip',
        'internal_submarket', 'internal_class', 'leasedSF', 'company_name',
        'internal_industry', 'transaction_type', 'internal_market_cluster',
        'costarID', 'space_type', 'CBD_suburban', 'RBA', 'available_space',
        'availability_proportion', 'internal_class_rent', 'overall_rent',
        'direct_available_space', 'direct_availability_proportion',
        'direct_internal_class_rent', 'direct_overall_rent',
        'sublet_available_space', 'sublet_availability_proportion',
        'sublet_internal_class_rent', 'sublet_overall_rent', 'leasing'],
       dtype='object'),
 (194685, 35))

In [13]:
# Columns with numbers
df_numeric = leaseData.select_dtypes(include=[np.number])
numberColumns = df_numeric.columns.tolist()

# Unique values in each column and checking if column has numerical values
def getUniques(data):
    uniqueCount = []
    for column in data.columns:
        unique_count = data[column].nunique()
        is_numeric = pd.api.types.is_numeric_dtype(data[column])
        nan_values = data[column].isna().sum()
        uniqueCount.append({"Column": column, "Unique_Count": unique_count, "hasNums": is_numeric, "NaN Values": nan_values, "Unique Values": data[column].unique()})

    numUniques = pd.DataFrame(uniqueCount)
    return numUniques

numUniques = getUniques(leaseData)
numUniques

,Column,Unique_Count,hasNums,NaN Values,Unique Values
0,year,7,False,0,"[2018, 2019, 2020, 2021, 2022, 2023, 2024]"
1,quarter,4,False,0,"[Q1, Q2, Q3, Q4]"
2,monthsigned,13,False,0,"[1.0, 2.0, 3.0, 4.0, 5.0, 12.0, nan, 7.0, 8.0,..."
3,market,29,False,0,"[Atlanta, Austin, Baltimore, Boston, Charlotte..."
4,building_name,14088,False,36686,"[10 Glenlake North Tower, 100 City View, 1000 ..."
5,building_id,21077,False,0,[Atlanta_Central Perimeter_Atlanta_10 Glenlake...
6,address,20636,False,0,"[10 Glenlake Pky NE, 3330 Cumberland Blvd, 100..."
7,region,4,False,0,"[South, Northeast, Midwest/Central, West]"
8,city,989,False,0,"[Atlanta, Peachtree Corners, Gainesville, Deca..."
9,state,22,False,0,"[GA, TX, MD, MA, NH, NC, IL, CO, MI, CA, NY, T..."


In [4]:
#Columns with less unique values might be more generalizable
lessUnique = []
for i, row in numUniques.iterrows():
    if row['Unique_Count'] < 40:
        column_name = row['Column']
        if row['hasNums']:
            leaseData.loc[:, column_name] = leaseData[column_name].astype(str)
        unique_values = leaseData[column_name].unique().tolist()
        lessUnique.append({"Column": column_name, "Unique_Values": unique_values})

generalizable = pd.DataFrame(lessUnique)
generalizable

,Column,Unique_Values
0,year,"[2018, 2019, 2020, 2021, 2022, 2023, 2024]"
1,quarter,"[Q1, Q2, Q3, Q4]"
2,monthsigned,"[1.0, 2.0, 3.0, 4.0, 5.0, 12.0, nan, 7.0, 8.0,..."
3,market,"[Atlanta, Austin, Baltimore, Boston, Charlotte..."
4,region,"[South, Northeast, Midwest/Central, West]"
5,state,"[GA, TX, MD, MA, NH, NC, IL, CO, MI, CA, NY, T..."
6,internal_class,"[A, O, nan]"
7,internal_industry,"[Financial Services and Insurance, nan, Constr..."
8,transaction_type,"[Expansion, New, Relocation, Renewal, Restruct..."
9,internal_market_cluster,"[nan, North Suburbs, South Suburbs, Baltimore ..."


### Now we can also analyze which of these columns we can drop \(either cut or combine extra information \)

In [5]:
specified_columns = ['market', 'city', 'building_name', 'address', 'internal_submarket']
existing_columns = [column for column in specified_columns if column in leaseData.columns]

for i in range(5):
    print("Existing Columns: \n", leaseData[existing_columns].iloc[i])
    print("Building_id:", leaseData['building_id'].iloc[i])


Existing Columns: 
 market                                Atlanta
city                                  Atlanta
building_name         10 Glenlake North Tower
address                    10 Glenlake Pky NE
internal_submarket          Central Perimeter
Name: 0, dtype: object
Building_id: Atlanta_Central Perimeter_Atlanta_10 Glenlake North Tower_10 Glenlake Pky NE
Existing Columns: 
 market                             Atlanta
city                               Atlanta
building_name                100 City View
address               3330 Cumberland Blvd
internal_submarket               Northwest
Name: 1, dtype: object
Building_id: Atlanta_Northwest_Atlanta_100 City View_3330 Cumberland Blvd
Existing Columns: 
 market                             Atlanta
city                               Atlanta
building_name                1000 Parkwood
address               1000 Parkwood Cir SE
internal_submarket               Northwest
Name: 2, dtype: object
Building_id: Atlanta_Northwest_Atlanta_1000 Par

In [6]:
for index, row in leaseData.iterrows():
    if row['city'] != row['market']:
        print(f"First mismatch found at index {index}:\nCity: {row['city']}\nMarket: {row['market']}\nBuilding_id: {row['building_id']}")
        break

similar_count = 0
mismatch_count = 0

for index, row in leaseData.iterrows():
    if row['city'] == row['market']:
        similar_count += 1
    else:
        mismatch_count += 1

print(f"Number of similar entries: {similar_count}")
print(f"Number of mismatched entries: {mismatch_count}")

First mismatch found at index 6:
City: Peachtree Corners
Market: Atlanta
Building_id: Atlanta_Northeast_Peachtree Corners_2 Sun_2 Sun Ct
Number of similar entries: 64990
Number of mismatched entries: 129695


In [7]:
dropColumns = ['building_name', 'market', 'city', 'address', 'internal_submarket', #buildingID already contains the following and also doesn't have NaN values
'CBD_suburban' #where the lease was signed, either Central Business District or Suburban, unnecessary information
]
dropData = leaseData.drop(columns=dropColumns)

We already tested to double check that available\_space = direct\_available\_space \+ sublet\_available\_space

Moreover, this can be condensed into 1008 unique ratios for direct:sublet

In [47]:
#offices with NaN available space, meaning it can't be rented
leaseDataNotRentable = dropData[dropData['available_space'].isna()]
# Drop all columns in leaseDataNotRentable starting from RBA
leaseDataNotRentable = leaseDataNotRentable.iloc[:, :15]

#rentable offices
leaseDataRentable = dropData.dropna(subset=['available_space'])
#rentable offices being subletted
leaseDataWithSublets = leaseDataRentable.dropna(subset=['direct_available_space'])
#rentable offices not being subletted
leaseDataNoSublets = leaseDataRentable[leaseDataRentable['direct_available_space'].isna()]

leaseDataNotRentable.shape, leaseDataRentable.shape, leaseDataWithSublets.shape, leaseDataNoSublets.shape

((49144, 15), (145541, 29), (121674, 29), (23867, 29))

In [49]:
getUniques(leaseDataNotRentable)

,Column,Unique_Count,hasNums,NaN Values,Unique Values
0,year,7,False,0,"[2018, 2019, 2020, 2021, 2022, 2023, 2024]"
1,quarter,4,False,0,"[Q1, Q2, Q3, Q4]"
2,monthsigned,13,False,0,"[1.0, 2.0, 3.0, 4.0, 5.0, 12.0, nan, 8.0, 11.0..."
3,building_id,4925,False,0,[Chicago_Central Loop_Chicago_11 E Adams_11 E ...
4,region,4,False,0,"[Midwest/Central, South, West, Northeast]"
5,state,8,False,0,"[IL, TX, CO, CA, NC, MD, DC, NY]"
6,zip,399,True,0,"[60603.0, 60601.0, 60602.0, 60606.0, 60607.0, ..."
7,internal_class,2,False,9,"[O, A, nan]"
8,leasedSF,14039,True,0,"[2992.0, 1894.0, 1500.0, 18206.0, 3838.0, 1526..."
9,company_name,7737,False,39864,"[nan, Salv, Tomasik Kotin & Kasserman, Roetzel..."


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=4487cead-9e45-4286-b5b8-37614f9dfcb9' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>